# Task 11, Notebook 01: Resume data audit

**Programme:** Aura Workforce by Growmar, AI/ML track, CortexHR **Phase 3: Intelligence**  
**Task:** 11: Rank resumes against job descriptions  
**Objective:** Understand and audit resumes, job descriptions and human labels before building anything.  
**Workflow stages covered in this notebook:** 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13 of the 38-stage Phase 1/2 standard.  
**Rule for every number below:** it is printed by the code in the same notebook; nothing is typed in by hand.

In [1]:
import sys, warnings, pathlib
warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / 'config' / 'phase3_config.yaml').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40); pd.set_option('display.max_colwidth', 80)
from cortexhr.phase3.common.config import load_config
from cortexhr.phase3.common import paths as P
CFG = load_config(); SEED = CFG['seed']
print('project root:', ROOT.name, '| seed:', SEED)

project root: phase3 | seed: 42


## Stage 1: Business Understanding

*What this stage is for:* Who needs this result and what decision it supports.

Recruiters read hundreds of resumes for every opening. A ranked shortlist saves time, but hiring is one of the most legally sensitive uses of AI, so the ranking must be explainable and must never use name, gender, age or college tier. Everything here is decision support: a human recruiter makes the decision.

## Stage 2: Problem Definition

*What this stage is for:* The exact prediction, ranking or measurement problem, its inputs and its output.

**Input:** one job description and a pool of resumes. **Output:** every resume gets a score for that job, candidates are ranked, the top 5 are returned with matched and missing skills. **Type:** ranking (not classification), evaluated against human suitability scores.

## Stage 3: Objective and Success Criteria

*What this stage is for:* What number or evidence shows the task is done well, decided before looking at results.

Success criteria are fixed here, before any result is seen (they live in `config/phase3_config.yaml`).

In [2]:
pd.Series(CFG['resume']['success_criteria'], name='threshold').to_frame()

                                       threshold
precision_at_5_min_multiple_of_random      2.000
hit_rate_at_5_min                          0.800
max_twin_gap_after_fix                     0.001
max_ndcg_drop_after_fix                    0.050

## Stage 4: Dataset Discovery

*What this stage is for:* Which files exist, how they relate, and which are authoritative.

Four tables are involved. `resumes` (candidates), `job_descriptions` (10 openings), `resume_job_labels` (human judgement for resume x job pairs) and `resume_bias_pairs` (25 twin resumes, used in Task 12).

In [3]:
from cortexhr.phase3.common.io import read_csv
from cortexhr.phase3.resume import text as T
from cortexhr.phase3.common.validation import profile
resumes = read_csv('resumes'); jobs = read_csv('job_descriptions'); labels = read_csv('resume_job_labels'); pairs = read_csv('resume_bias_pairs')
dd = read_csv('data_dictionary')
print('resumes', resumes.shape, '| jobs', jobs.shape, '| labels', labels.shape, '| twin pairs', pairs.shape)

resumes (185, 15) | jobs (10, 11) | labels (555, 7) | twin pairs (25, 8)


## Stage 5: Data Loading

*What this stage is for:* Load with the project helpers and check the load is complete.

Loaded with the shared reader, which raises a clear error if a file is missing.

In [4]:
{k: v.shape for k, v in dict(resumes=resumes, jobs=jobs, labels=labels, pairs=pairs).items()}

{'resumes': (185, 15), 'jobs': (10, 11), 'labels': (555, 7), 'pairs': (25, 8)}

## Stage 6: Dataset Structure and Shape

*What this stage is for:* Rows, columns, keys and grain of each table.

Grain: `resumes` = one row per resume; `jobs` = one row per job; `labels` = one row per (resume, job) pair.

In [5]:
print('resume_id unique:', resumes.resume_id.is_unique, '| job_id unique:', jobs.job_id.is_unique, '| (resume,job) unique in labels:', not labels.duplicated(['resume_id','job_id']).any())
print('labelled jobs per resume:', labels.groupby('resume_id').size().value_counts().to_dict())
print('labelled resumes per job:'); print(labels.groupby('job_id').size().describe().round(1).to_dict())
resumes.columns.tolist()

resume_id unique: True | job_id unique: True | (resume,job) unique in labels: True
labelled jobs per resume: {3: 185}
labelled resumes per job:
{'count': 10.0, 'mean': 55.5, 'std': 5.7, 'min': 47.0, '25%': 50.5, '50%': 56.0, '75%': 59.2, 'max': 64.0}


['resume_id', 'candidate_name', 'email', 'city', 'total_experience_years', 'current_title', 'current_company', 'highest_education', 'college_tier', 'skills', 'certifications', 'source_format', 'layout_difficulty', 'target_job_family', 'resume_text']

## Stage 7: Data Dictionary and Schema Understanding

*What this stage is for:* What each column means, from the supplied dictionary and from inspection.

From the supplied data dictionary and from reading the data.

In [6]:
dd[dd.sheet.str.contains('resume|job', case=False)][['sheet', 'column', 'data_type', 'description']].head(40)

                 sheet                     column data_type                                                                   description
191   job_descriptions                     job_id      text                                                             Job id (J01 ...).
192   job_descriptions                  job_title      text                                                                    Job title.
193   job_descriptions              department_id      text                   Department code (D00 Executive ... D10 Legal & Compliance).
194   job_descriptions                  job_level   integer                                 Level 1 to 7 (1 Associate ... 6 Head, 7 CEO).
195   job_descriptions       min_experience_years   integer                                                  Minimum experience required.
196   job_descriptions            required_skills      text                                       Must-have skills (semicolon separated).
197   job_descriptions        nice

## Stage 8: Data Type Inspection

*What this stage is for:* Check every column has the type it should have.

Every column should have the type its meaning implies.

In [7]:
pd.concat({'resumes': resumes.dtypes.astype(str), 'jobs': jobs.dtypes.astype(str), 'labels': labels.dtypes.astype(str)})

resumes  resume_id                      str
         candidate_name                 str
         email                          str
         city                           str
         total_experience_years     float64
         current_title                  str
         current_company                str
         highest_education              str
         college_tier                   str
         skills                         str
         certifications                 str
         source_format                  str
         layout_difficulty              str
         target_job_family              str
         resume_text                    str
jobs     job_id                         str
         job_title                      str
         department_id                  str
         job_level                    int64
         min_experience_years         int64
         required_skills                str
         nice_to_have_skills            str
         education_preference   

## Stage 9: Data Quality Assessment

*What this stage is for:* A one-page quality profile before any modelling.

One-page quality profile per table.

In [8]:
profile(resumes).join(profile(resumes.select_dtypes('number')).add_prefix('num_'), how='left').head(20)

                          dtype  missing  missing_pct  unique num_dtype  num_missing  num_missing_pct  num_unique
resume_id                   str        0         0.00     185       NaN          NaN              NaN         NaN
candidate_name              str        0         0.00     180       NaN          NaN              NaN         NaN
email                       str        0         0.00     180       NaN          NaN              NaN         NaN
city                        str        0         0.00      10       NaN          NaN              NaN         NaN
total_experience_years  float64        0         0.00      73   float64          0.0              0.0        73.0
current_title               str        0         0.00      34       NaN          NaN              NaN         NaN
current_company             str        0         0.00      12       NaN          NaN              NaN         NaN
highest_education           str        0         0.00      16       NaN          NaN    

## Stage 10: Missing Value Analysis

*What this stage is for:* Where values are missing, how much, and why that may be.

Missing values, including empty or blank free text.

In [9]:
print('resumes with missing values per column:'); print(resumes.isna().sum()[lambda s: s > 0].to_dict() or 'none')
print('blank resume_text:', int((resumes.resume_text.fillna('').str.strip() == '').sum()), '| blank skills:', int((resumes.skills.fillna('').str.strip() == '').sum()))
print('jobs missing values:', jobs.isna().sum()[lambda s: s > 0].to_dict() or 'none')
print('labels missing values:', labels.isna().sum()[lambda s: s > 0].to_dict() or 'none')

resumes with missing values per column:
{'certifications': 114}
blank resume_text: 0 | blank skills: 0
jobs missing values: none
labels missing values: none


## Stage 11: Duplicate Analysis

*What this stage is for:* Exact and near duplicates of rows, keys and texts.

Exact duplicates, and near duplicates. The 25 twin resumes are near duplicates **by design** (same content, one detail changed): they must be recognised, not removed.

In [10]:
from cortexhr.phase3.resume import text as T
print('duplicate resume_id:', int(resumes.resume_id.duplicated().sum()), '| exact duplicate resume_text:', int(resumes.resume_text.duplicated().sum()))
masked = resumes.apply(lambda r: T.normalise(T.mask_identifiers(r.resume_text, r.candidate_name, r.email, r.city)), axis=1)
dup_groups = resumes.assign(masked=masked).groupby('masked').resume_id.apply(list); dup_groups = dup_groups[dup_groups.str.len() > 1]
twin_ids = set(pairs.resume_id_original) | set(pairs.resume_id_twin)
print('near-duplicate groups after masking identifiers:', len(dup_groups), '| all explained by the twin pairs:', all(set(g) <= twin_ids for g in dup_groups))
print('duplicate job descriptions:', int(jobs.description_text.duplicated().sum()), '| duplicate job titles:', int(jobs.job_title.duplicated().sum()))

duplicate resume_id: 0 | exact duplicate resume_text: 0
near-duplicate groups after masking identifiers: 25 | all explained by the twin pairs: True
duplicate job descriptions: 0 | duplicate job titles: 0


## Stage 12: Outlier Detection and Investigation

*What this stage is for:* Unusual values: are they errors or real?

Outliers in the numeric fields and in text length. An outlier here is only a flag to look at, not something to delete.

In [11]:
x = resumes.total_experience_years; q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
out = resumes[(x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)]
print('experience IQR outliers:', len(out), '| max years:', x.max(), '| z>3:', int((((x - x.mean()) / x.std()).abs() > 3).sum()))
L = resumes.resume_text.str.len(); print('text length min/median/max:', L.min(), int(L.median()), L.max())
resumes.assign(text_len=L).sort_values('text_len').head(3)[['resume_id', 'layout_difficulty', 'text_len']]

experience IQR outliers: 0 | max years: 9.3 | z>3: 0
text length min/median/max: 437 607 1599


    resume_id layout_difficulty  text_len
31   RES-0032  Missing sections       437
164  RES-0165  Missing sections       444
117  RES-0118  Missing sections       452

## Stage 13: Invalid and Inconsistent Value Detection

*What this stage is for:* Impossible values, broken links between tables, label contradictions.

Broken links, impossible values and label contradictions.

In [12]:
from cortexhr.phase3.common.validation import require_subset
require_subset(labels.resume_id, resumes.resume_id, 'label resume_ids'); require_subset(labels.job_id, jobs.job_id, 'label job_ids')
require_subset(pairs.resume_id_original, resumes.resume_id); require_subset(pairs.resume_id_twin, resumes.resume_id)
checks = {'all label resume ids exist': True, 'all label job ids exist': True,
          'human score in 1..5': bool(labels.human_suitability_score.between(1, 5).all()),
          'shortlist == (score >= 4)': bool(((labels.human_suitability_score >= 4) == (labels.shortlist == 'Y')).all()),
          'experience between 0 and 40': bool(resumes.total_experience_years.between(0, 40).all()),
          'job minimum experience >= 0': bool((jobs.min_experience_years >= 0).all())}
pd.Series(checks, name='passed').to_frame()

                             passed
all label resume ids exist     True
all label job ids exist        True
human score in 1..5            True
shortlist == (score >= 4)      True
experience between 0 and 40    True
job minimum experience >= 0    True

## Findings of this audit

Printed from the checks above: no broken links, labels are internally consistent, and the only near duplicates are the planned twin pairs. These facts are used in every later notebook.

In [13]:
print('Audit conclusion:', 'PASS' if all(checks.values()) else 'FAIL', '|', sum(checks.values()), 'of', len(checks), 'checks passed;', len(dup_groups), 'twin near-duplicate groups found.')

Audit conclusion: PASS | 6 of 6 checks passed; 25 twin near-duplicate groups found.
